# Trading Agent — Setup & Data Download

**Run this notebook once per year to download that year's data.**

| Section | What it does |
|---|---|
| 1 | Check environment (packages, .env) |
| 2 | Login to Kite Connect |
| 3 | Build universe (Nifty 500 instrument tokens) |
| 4 | Download 1 year of 5-min data |
| 5 | Verify & review downloaded data |

**Learning phase:** 2016 → 2022 (run once per year, 7 sessions total)  
**Testing phase:** 2023 → 2026 YTD (weights frozen after 2022)

## SECTION 1 — Environment Check

In [2]:
1+1

2

In [4]:
import importlib, sys
from pathlib import Path

# Add project root to path so all imports work from notebooks/
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

required = ['kiteconnect', 'pandas', 'pyarrow', 'pandas_ta',
            'yfinance', 'dotenv', 'tqdm', 'plotly', 'numpy', 'curl_cffi']

missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f'  OK   {pkg}')
    except ImportError:
        print(f'  MISSING  {pkg}')
        missing.append(pkg)

if missing:
    print(f'\nInstall missing: pip install {" ".join(missing)}')
else:
    print('\nAll packages OK.')

  OK   kiteconnect
  OK   pandas
  OK   pyarrow
  OK   pandas_ta
  OK   yfinance
  OK   dotenv
  OK   tqdm
  OK   plotly
  OK   numpy
  OK   curl_cffi

All packages OK.


In [5]:
import os
from dotenv import load_dotenv

load_dotenv(project_root / '.env')
api_key    = os.getenv('KITE_API_KEY')
api_secret = os.getenv('KITE_API_SECRET')

print(f'API Key    : {api_key[:8]}...  ({"OK" if api_key else "MISSING"})')
print(f'API Secret : {api_secret[:8]}...  ({"OK" if api_secret else "MISSING"})')

from config.settings import *
print(f'\nLearning phase : {LEARNING_START_YEAR} – {LEARNING_END_YEAR}')
print(f'Testing phase  : {TESTING_START_YEAR} – {TESTING_END_YEAR}')

API Key    : 5yqpo83g...  (OK)
API Secret : qt9majb6...  (OK)

Learning phase : 2016 – 2022
Testing phase  : 2023 – 2026


## SECTION 2 — Login to Kite Connect

**Step 1:** Run the cell below to get the login URL → paste in browser → log in  
**Step 2:** Copy `request_token` from the redirect URL bar (even if page shows an error)  
**Step 3:** Paste into the next cell and run it

The redirect URL looks like:  
`127.0.0.1/?status=success&request_token=XXXXX&action=login`  
The page may show 'site can't be reached' — that is normal. Copy the token from the URL bar.

In [3]:
from data_pipeline.kite_auth import login_step1
login_step1()

STEP 1: Open this URL in your browser and log in:

https://kite.zerodha.com/connect/login?api_key=5yqpo83ga1pfxe3a&v=3

After login you'll be redirected to a URL like:
  https://127.0.0.1/?request_token=XXXXX&action=login&status=success

Copy the request_token value and pass it to login_step2()


'https://kite.zerodha.com/connect/login?api_key=5yqpo83ga1pfxe3a&v=3'

In [13]:
# Paste your request_token from the redirect URL here
REQUEST_TOKEN = "AgTR6Spf7Gz58quC8HzVOeGJaxXpuurl"

from data_pipeline.kite_auth import login_step2
kite = login_step2(REQUEST_TOKEN)
print('Kite Connect ready.')

Login successful. Access token saved for 2026-06-10.
You can now use get_kite() for the rest of this session.
Kite Connect ready.


In [14]:
# Already logged in today? Load cached token (skips browser login)
from data_pipeline.kite_auth import get_kite
try:
    kite = get_kite()
    profile = kite.profile()
    print(f'Logged in as : {profile["user_name"]} ({profile["email"]})')
    print(f'Broker       : {profile["broker"]}')
except Exception as e:
    print(f'Not logged in: {e}')
    print('Run the login cells (c3 and c4) first.')

Logged in as : Somal Kant (somalkant.mnnit@gmail.com)
Broker       : ZERODHA


## SECTION 3 — Build Universe (Nifty 500 Instrument Tokens)

Fetches the Nifty 500 symbol list from NSE (tries 3 methods automatically),
then matches each symbol with its Kite instrument token.  
Saves to `config/universe.csv`. **Run once — persists across all sessions.**

In [15]:
import pandas as pd

# Step 1: Get all NSE instruments from Kite (includes tokens for every stock)
print('Fetching NSE instrument list from Kite Connect...')
instruments = kite.instruments('NSE')
inst_df = pd.DataFrame(instruments)
print(f'Total NSE instruments : {len(inst_df)}')
inst_df[inst_df['instrument_type'] == 'EQ'][['instrument_token','tradingsymbol','name']].head(5)

Fetching NSE instrument list from Kite Connect...
Total NSE instruments : 9812


,instrument_token,tradingsymbol,name
0,256265,NIFTY 50,NIFTY 50
1,256777,NIFTY MIDCAP 100,NIFTY MIDCAP 100
2,260105,NIFTY BANK,NIFTY BANK
3,260617,NIFTY 100,NIFTY 100
4,257033,NIFTY DIV OPPS 50,NIFTY DIV OPPS 50


In [16]:
# Step 2: Fetch Nifty 500 symbol list
# Tries 3 methods automatically:
#   1. NSE Archives direct CSV (most reliable — no auth needed)
#   2. curl_cffi browser impersonation
#   3. requests with session cookies
#   4. Local cache from a previous run

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

from data_pipeline.universe import fetch_nifty500_symbols

nifty500_symbols = fetch_nifty500_symbols()
print(f'\nNifty 500 symbols fetched : {len(nifty500_symbols)}')
print(f'First 10 : {nifty500_symbols[:10]}')

INFO: Method 1 (NSE Archive CSV): fetched 504 symbols
INFO: Saved 504 symbols to C:\Users\somal\Downloads\AI_ML\TradingAgent\config\nifty500_symbols.csv



Nifty 500 symbols fetched : 504
First 10 : ['360ONE', '3MINDIA', 'ABB', 'ACC', 'ACMESOLAR', 'AIAENG', 'APLAPOLLO', 'AUBANK', 'AWL', 'AADHARHFC']


In [17]:
# Step 3: Match symbols to Kite instrument tokens and save universe.csv
eq_df = inst_df[
    (inst_df['exchange'] == 'NSE') &
    (inst_df['instrument_type'] == 'EQ') &
    (inst_df['tradingsymbol'].isin(nifty500_symbols))
][['instrument_token', 'tradingsymbol', 'name', 'lot_size', 'tick_size']].copy()

eq_df = eq_df.drop_duplicates(subset=['tradingsymbol'])
eq_df = eq_df.sort_values('tradingsymbol').reset_index(drop=True)

print(f'Matched {len(eq_df)} / {len(nifty500_symbols)} symbols with Kite tokens')
if len(eq_df) < 450:
    print(f'WARNING: Only {len(eq_df)} stocks matched. Some symbols may differ between NSE and Kite naming.')

universe_path = project_root / 'config' / 'universe.csv'
eq_df.to_csv(universe_path, index=False)
print(f'Saved → {universe_path}')
eq_df.head(10)

Matched 500 / 504 symbols with Kite tokens
Saved → C:\Users\somal\Downloads\AI_ML\TradingAgent\config\universe.csv


,instrument_token,tradingsymbol,name,lot_size,tick_size
0,3343617,360ONE,360 ONE WAM,1,0.10
1,121345,3MINDIA,3M INDIA,1,5.00
2,6074625,AADHARHFC,AADHAR HOUSING FINANCE L,1,0.05
3,1793,AARTIIND,AARTI INDUSTRIES,1,0.05
4,1378561,AAVAS,AAVAS FINANCIERS,1,0.10
5,3329,ABB,ABB INDIA,1,0.50
6,4583169,ABBOTINDIA,ABBOTT INDIA,1,5.00
7,5533185,ABCAPITAL,ADITYA BIRLA CAPITAL,1,0.05
8,6222849,ABDL,ALLIED BLEND N DISTILS L,1,0.05
9,7707649,ABFRL,ADITYA BIRLA FASHION & RT,1,0.01


## SECTION 4 — Download Historical Data

Set `YEAR_TO_DOWNLOAD` and run. The downloader:
- Splits each year into 95-day chunks (Kite limit: 100 days per request)
- Rate-limits to 3 req/sec automatically
- Saves progress after every stock — **safe to stop and resume**
- Appends to Parquet files — previous years are never overwritten

**Learning:** 2016, 2017, 2018, 2019, 2020, 2021, 2022  
**Testing:**  2023, 2024, 2025, 2026

In [25]:
# ── SET THIS ──
YEAR_TO_DOWNLOAD = 2026
# ──────────────

from config.settings import get_phase
phase = get_phase(YEAR_TO_DOWNLOAD)
print(f'Year   : {YEAR_TO_DOWNLOAD}')
print(f'Phase  : {phase}')
print(f'Action : {"Build strategy weights" if phase == "LEARNING" else "Test frozen weights"}')
print()
print('Download will start now. Safe to interrupt — resumes from last completed stock.')

Year   : 2026
Phase  : TESTING
Action : Test frozen weights

Download will start now. Safe to interrupt — resumes from last completed stock.


In [26]:
from data_pipeline.downloader import download_year

result = download_year(kite, year=YEAR_TO_DOWNLOAD)

print(f'\nYear {YEAR_TO_DOWNLOAD} complete:')
print(f'  Stocks downloaded : {result["completed"]}')
print(f'  Failed            : {len(result["failed"])}')
if result['failed']:
    print(f'  Failed symbols    : {result["failed"]}')
    print('  → Run: retry_failed(kite, YEAR_TO_DOWNLOAD) to retry')

INFO: Downloading year 2026 — 0 stocks to process
INFO: Date range: 2026-01-01  →  2026-12-31
Year 2026: 0stock [00:00, ?stock/s]
INFO: Downloading NIFTY50 index data...
INFO: Downloading INDIA VIX data...
INFO: Year 2026 done: 500 completed, 0 failed



Year 2026 complete:
  Stocks downloaded : 500
  Failed            : 0


In [27]:
# Check download status across all years
from data_pipeline.downloader import get_download_status
from config.settings import get_phase

status = get_download_status()
if not status:
    print('No downloads recorded yet.')
else:
    print(f'{"Year":<6} {"Phase":<10} {"Done":<6} {"Failed":<8} {"Pct":<7} {"Status"}')
    print('-' * 55)
    for yr, s in sorted(status.items()):
        phase_label = get_phase(int(yr))
        print(f'{yr:<6} {phase_label:<10} {s["completed"]:<6} {s["failed"]:<8} {s["pct"]:<7} {s["status"]}')

Year   Phase      Done   Failed   Pct     Status
-------------------------------------------------------
2016   LEARNING   334    166      66.8%   partial
2017   LEARNING   350    150      70.0%   partial
2018   LEARNING   368    132      73.6%   partial
2019   LEARNING   377    123      75.4%   partial
2020   LEARNING   385    115      77.0%   partial
2021   LEARNING   414    86       82.8%   partial
2022   LEARNING   429    103      85.8%   partial
2023   TESTING    446    71       89.2%   partial
2024   TESTING    471    29       94.2%   partial
2025   TESTING    500    0        100.0%  completed
2026   TESTING    500    0        100.0%  completed


## SECTION 5 — Verify & Review Downloaded Data

In [28]:
import pandas as pd
from config.settings import STOCKS_DIR

year_dir = STOCKS_DIR / str(YEAR_TO_DOWNLOAD)
parquet_files = sorted(year_dir.glob("*.parquet")) if year_dir.exists() else []
print(f"Year folder       : {year_dir}")
print(f"Stocks downloaded : {len(parquet_files)}")

stats = []
for f in parquet_files[:25]:
    df = pd.read_parquet(f)
    stats.append({
        "symbol"  : f.stem,
        "candles" : len(df),
        "from"    : str(df["datetime"].min())[:10],
        "to"      : str(df["datetime"].max())[:10],
        "size_KB" : round(f.stat().st_size / 1024, 1),
    })

total_mb = sum(f.stat().st_size for f in parquet_files) / 1024**2
print(f"Total storage     : {total_mb:.1f} MB")
pd.DataFrame(stats)

Year folder       : C:\Users\somal\Downloads\AI_ML\TradingAgent\data\stocks\2026
Stocks downloaded : 500
Total storage     : 99.2 MB


,symbol,candles,from,to,size_KB
0,360ONE,7975,2026-01-01,2026-06-09,198.6
1,3MINDIA,7952,2026-01-01,2026-06-09,150.4
2,AADHARHFC,7955,2026-01-01,2026-06-09,170.0
3,AARTIIND,7956,2026-01-01,2026-06-09,199.2
4,AAVAS,7955,2026-01-01,2026-06-09,190.1
5,ABB,7975,2026-01-01,2026-06-09,210.0
6,ABBOTINDIA,7955,2026-01-01,2026-06-09,135.4
7,ABCAPITAL,7975,2026-01-01,2026-06-09,185.8
8,ABDL,7955,2026-01-01,2026-06-09,201.5
9,ABFRL,7956,2026-01-01,2026-06-09,195.2


In [22]:
# Preview candles for any stock
PREVIEW_STOCK = "RELIANCE"

parquet_path = year_dir / f"{PREVIEW_STOCK}.parquet"
if not parquet_path.exists():
    print(f"File not found: {parquet_path}")
    print("Run the download cell (c10) first, or pick a stock that was downloaded.")
else:
    df = pd.read_parquet(parquet_path)
    df["datetime"] = pd.to_datetime(df["datetime"])

    first_day = df["datetime"].dt.date.min()
    day_data  = df[df["datetime"].dt.date == first_day]

    print(f"{PREVIEW_STOCK}: {len(df):,} total candles")
    print(f"Range: {df['datetime'].min().date()} to {df['datetime'].max().date()}")
    print(f"\nFirst trading day ({first_day}) -- {len(day_data)} candles:")
    day_data[["datetime","open","high","low","close","volume"]].head(10)

RELIANCE: 7,950 total candles
Range: 2026-01-01 to 2026-06-09

First trading day (2026-01-01) -- 75 candles:


In [23]:
# Data quality — count missing trading days
dates        = sorted(df['datetime'].dt.date.unique())
all_weekdays = pd.date_range(str(dates[0]), str(dates[-1]), freq='B')
date_set     = set(str(d) for d in dates)
missing      = [d for d in all_weekdays if str(d.date()) not in date_set]

print(f'Trading days in data : {len(dates)}')
print(f'Weekdays in range    : {len(all_weekdays)}')
print(f'Missing days         : {len(missing)}  (NSE holidays + any API gaps)')
print('Data quality: OK' if len(missing) < 30 else 'WARNING: many missing days')

Trading days in data : 106
Weekdays in range    : 114
Missing days         : 9  (NSE holidays + any API gaps)
Data quality: OK


In [24]:
# Interactive candlestick chart
import plotly.graph_objects as go

CHART_STOCK = "RELIANCE"
CHART_DATE  = str(df["datetime"].dt.date.min())

year_dir = STOCKS_DIR / str(YEAR_TO_DOWNLOAD)
chart_df = pd.read_parquet(year_dir / f"{CHART_STOCK}.parquet")
chart_df["datetime"] = pd.to_datetime(chart_df["datetime"])
day = chart_df[chart_df["datetime"].dt.date.astype(str) == CHART_DATE]

fig = go.Figure(go.Candlestick(
    x=day["datetime"], open=day["open"], high=day["high"],
    low=day["low"], close=day["close"], name=CHART_STOCK
))
fig.add_bar(
    x=day["datetime"], y=day["volume"], name="Volume",
    yaxis="y2", marker_color="rgba(100,149,237,0.4)"
)
fig.update_layout(
    title=f"{CHART_STOCK} - 5-min candles - {CHART_DATE}",
    xaxis_rangeslider_visible=False,
    yaxis=dict(title="Price (Rs)", domain=[0.25, 1.0]),
    yaxis2=dict(title="Volume", domain=[0.0, 0.20], anchor="x"),
    height=600, template="plotly_dark"
)
fig.show()

## SECTION 6 — Daily Update (run every day after market close)

Downloads only the **last 7 days** for all 500 stocks — much faster than a full year download (~10 min).  
Run this each evening after 3:30 PM to keep parquet files current for tomorrow's live session.  
Safe to run multiple times — duplicates are removed automatically.

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import time
from datetime import date, timedelta
from pathlib import Path
from tqdm import tqdm
from config.settings import STOCKS_DIR, INDEX_DIR, DATA_INTERVAL, CHUNK_DAYS, RATE_LIMIT_SLEEP, MAX_RETRIES, RETRY_BACKOFF, UNIVERSE_FILE

def _safe_append(path: Path, df: pd.DataFrame) -> None:
    """Append df to parquet. Always casts OHLC to float to avoid schema mismatch."""
    for col in ["open", "high", "low", "close"]:
        df[col] = df[col].astype(float)
    df["volume"] = df["volume"].astype("int64")
    if path.exists():
        existing_df = pq.read_table(path).to_pandas()
        merged = (pd.concat([existing_df, df])
                  .drop_duplicates(subset=["datetime"])
                  .sort_values("datetime")
                  .reset_index(drop=True))
        for col in ["open", "high", "low", "close"]:
            merged[col] = merged[col].astype(float)
        merged["volume"] = merged["volume"].astype("int64")
        pq.write_table(pa.Table.from_pandas(merged, preserve_index=False), path, compression="snappy")
    else:
        pq.write_table(pa.Table.from_pandas(df, preserve_index=False), path, compression="snappy")

def update_recent(kite, days=7):
    """Download last N days for all 500 stocks. Safe to run daily."""
    universe = pd.read_csv(UNIVERSE_FILE)
    today    = date.today()
    from_dt  = today - timedelta(days=days)
    yr_dir   = STOCKS_DIR / str(today.year)
    yr_dir.mkdir(exist_ok=True)
    print(f"Updating {len(universe)} stocks: {from_dt} → {today}")

    failed = []
    for _, row in tqdm(universe.iterrows(), total=len(universe), desc="Daily update"):
        symbol = row["tradingsymbol"]
        token  = int(row["instrument_token"])

        # Build date chunks (same logic as downloader)
        chunks, cur = [], from_dt
        while cur <= today:
            end = min(cur + timedelta(days=CHUNK_DAYS - 1), today)
            chunks.append((cur, end))
            cur = end + timedelta(days=1)

        all_rows, ok = [], True
        for c_from, c_to in chunks:
            for attempt in range(MAX_RETRIES):
                try:
                    time.sleep(RATE_LIMIT_SLEEP)
                    raw = kite.historical_data(token, c_from, c_to, DATA_INTERVAL)
                    if raw:
                        all_rows.extend(raw)
                    break
                except Exception as e:
                    wait = RETRY_BACKOFF[attempt] if attempt < len(RETRY_BACKOFF) else 60
                    time.sleep(wait)
            else:
                ok = False
                break

        if not ok:
            failed.append(symbol)
            continue
        if not all_rows:
            continue

        df = pd.DataFrame(all_rows)
        df.rename(columns={"date": "datetime"}, inplace=True)
        df["symbol"] = symbol
        df = df[["datetime", "symbol", "open", "high", "low", "close", "volume"]]
        df["datetime"] = pd.to_datetime(df["datetime"])
        df.sort_values("datetime", inplace=True)
        df.drop_duplicates(subset=["datetime"], inplace=True)
        _safe_append(yr_dir / f"{symbol}.parquet", df)

    # Update NIFTY50 + INDIAVIX
    idx_dir = INDEX_DIR / str(today.year)
    idx_dir.mkdir(exist_ok=True)
    for name, token, fname in [("NIFTY 50", 256265, "NIFTY50"), ("INDIA VIX", 264969, "INDIAVIX")]:
        try:
            time.sleep(RATE_LIMIT_SLEEP)
            raw = kite.historical_data(token, from_dt, today, DATA_INTERVAL)
            if raw:
                df = pd.DataFrame(raw)
                df.rename(columns={"date": "datetime"}, inplace=True)
                df["symbol"] = fname
                df = df[["datetime", "symbol", "open", "high", "low", "close", "volume"]]
                df["datetime"] = pd.to_datetime(df["datetime"])
                df.sort_values("datetime", inplace=True)
                df.drop_duplicates(subset=["datetime"], inplace=True)
                _safe_append(idx_dir / f"{fname}.parquet", df)
        except Exception as e:
            print(f"  {name} failed: {e}")

    print(f"\nDone. Updated through {today}.")
    if failed:
        print(f"Failed ({len(failed)}): {failed}")
    else:
        print("All stocks updated successfully.")

update_recent(kite, days=7)